# CPU restore of completed Kaggle shard0008
Attach private payload v1 and private shard0008 Dataset v1. Restore the exact original runtime plan, run only --verify-only, then recompute saved Development metrics. No GPU, training or Final labels.

In [ ]:
import hashlib
import importlib.util
import json
import time
import types
from pathlib import Path

import torch

assert not torch.cuda.is_available(), "CPU restore verifier only"
MODEL_CODE = "44b8532959d76fb20691881539f991ca44983727"
PAYLOAD_SHA = "23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4"
HELPER_SHA = "cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424"
payloads = list(Path("/kaggle/input").rglob("provider-payload.zip.bin"))
archives = list(Path("/kaggle/input").rglob("artifacts.zip.bin"))
assert len(payloads) == len(archives) == 1, "Attach only pinned payload v1 and shard0008 Dataset v1"
payload, archive = payloads[0], archives[0]
helper = payload.parent / "hung_provider.py"
assert hashlib.sha256(helper.read_bytes()).hexdigest() == HELPER_SHA
assert (
    hashlib.sha256(archive.read_bytes()).hexdigest()
    == "a49fc9c6c56b5e6cdcf54bb1af15d05cb511d017f3be3cb4abfb41f17dc151e8"
)
module_spec = importlib.util.spec_from_file_location("hung_provider", helper)
provider = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(provider)
base = Path("/kaggle/working/hung-matched-group03-v1")
started = time.perf_counter()
repo, plan_path = provider.materialize(payload, PAYLOAD_SHA, base)
assert (
    provider.sha256(plan_path) == "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4"
)
assert (
    provider.sha256(repo / ".cache/provider/shards.json")
    == "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342"
)
derivation = json.loads((base / "derivation.json").read_bytes())
assert (
    derivation["parent_plan_sha256"]
    == "2d448d2584f65baf7e09445e013b4283314b08ea2cce855c6850e9fc12ba87ee"
)
assert (
    derivation["parent_shards_sha256"]
    == "7dd340b84a51abca9dcb2e83490d088c07178ffbdbb9d81d68225d665a279c7f"
)

In [ ]:
RECEIVER_SOURCE = '"""Restore one completed Development shard, verify receipts and score on CPU."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\n\n\ndef require(value: bool, message: str) -> None:\n    if not value:\n        raise ValueError(message)\n\n\ndef verification_command(plan: Path, partition: Path, shard_id: str, sha256) -> list[str]:\n    """The receiver has no stage-execution command or execution fallback."""\n    return [\n        sys.executable,\n        "-u",\n        "-m",\n        "experiments.hung_sprint2.dispatch_matched",\n        "--plan",\n        str(plan),\n        "--plan-sha256",\n        sha256(plan),\n        "--shards",\n        str(partition),\n        "--shards-sha256",\n        sha256(partition),\n        "--shard-id",\n        shard_id,\n        "--resume",\n        "--verify-only",\n    ]\n\n\ndef restore(\n    provider,\n    archive: Path,\n    receipt_path: Path,\n    plan_path: Path,\n    partition_path: Path,\n    *,\n    root: Path,\n    expected_code: str,\n) -> dict:\n    """Validate the archive and exact runtime plan before creating any output."""\n    plan = json.loads(plan_path.read_bytes())\n    partition = json.loads(partition_path.read_bytes())\n    receipt = json.loads(receipt_path.read_bytes())\n    require(\n        plan["scope"] == "development_only" and plan["code_commit"] == expected_code,\n        "Development code identity differs",\n    )\n    plan_sha, partition_sha = provider.sha256(plan_path), provider.sha256(partition_path)\n    require(partition["plan_sha256"] == plan_sha == receipt["plan_sha256"], "Plan SHA differs")\n    require(partition_sha == receipt["shards_sha256"], "Partition SHA differs")\n    require(\n        receipt["exit_code"] == receipt["local_verify_exit_code"] == 0, "Execution/verifier failed"\n    )\n    require(receipt["crc"] == "passed", "Original archive CRC failed")\n    shard = next(s for s in partition["shards"] if s["id"] == receipt["shard_id"])\n    jobs = shard["job_ids"]\n    require(0 < len(jobs) <= 20 and len(jobs) == len(set(jobs)), "Not one bounded shard")\n    require(receipt["job_ids"] == jobs, "Receipt job set differs")\n    require(\n        archive.stat().st_size == receipt["artifact_bytes"]\n        and provider.sha256(archive) == receipt["artifact_sha256"],\n        "Archive SHA/size differs",\n    )\n    output = Path(plan["output_root"])\n    require(output.resolve().is_relative_to(root.resolve()), "Output outside receiver root")\n    require(not output.exists(), "Restore output already exists; no overwrite")\n    with zipfile.ZipFile(archive) as stream:\n        names = provider.safe_members(stream)\n        require(\n            json.loads(stream.read("dispatch_identity.json"))["plan_sha256"] == plan_sha,\n            "Archive dispatch identity differs",\n        )\n        require(\n            all(\n                n == "dispatch_identity.json"\n                or (\n                    len(n.split("/")) >= 3\n                    and n.split("/")[0] in {"jobs", "receipts", "logs"}\n                    and n.split("/")[1] in jobs\n                )\n                for n in names\n            ),\n            "Foreign archive member",\n        )\n        output.mkdir(parents=True, exist_ok=False)\n        stream.extractall(output)\n    return receipt\n\n\ndef receive(\n    provider,\n    scorer: dict,\n    archive: Path,\n    receipt_path: Path,\n    repo: Path,\n    *,\n    expected_code: str,\n    evidence: Path,\n    scorer_sha256: str,\n) -> dict:\n    """Verify all official stages and dependencies before recomputing live forecasts."""\n    root = repo.parent\n    require(evidence.resolve().is_relative_to(root.resolve()), "Evidence outside receiver")\n    require(not evidence.exists(), "Receiver evidence already exists")\n    plan = repo / ".cache/provider/plan.json"\n    partition = repo / ".cache/provider/shards.json"\n    started = time.perf_counter()\n    receipt = restore(\n        provider, archive, receipt_path, plan, partition, root=root, expected_code=expected_code\n    )\n    evidence.mkdir(exist_ok=False)\n    command = verification_command(plan, partition, receipt["shard_id"], provider.sha256)\n    environment = {\n        **os.environ,\n        "PYTHONUTF8": "1",\n        "PYTHONPATH": os.pathsep.join((str(repo / "src"), str(repo))),\n    }\n    verify_started = time.perf_counter()\n    with (evidence / "verify.log").open("x", encoding="utf-8") as log:\n        verified = subprocess.run(\n            command,\n            cwd=repo,\n            env=environment,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            timeout=1800,\n            check=False,\n        )\n    terminal = {\n        "exit_code": verified.returncode,\n        "command": command,\n        "wall_seconds": time.perf_counter() - verify_started,\n        "executed": False,\n        "purpose": "Official --verify-only; no training fallback",\n        "log_sha256": provider.sha256(evidence / "verify.log"),\n    }\n    provider.save(evidence / "verify-exit.json", terminal)\n    require(verified.returncode == 0, "CPU official receipt/dependency verification failed")\n    spec = {\n        "plan": {"path": str(plan), "sha256": provider.sha256(plan)},\n        "partition": {"path": str(partition), "sha256": provider.sha256(partition)},\n        "completed_shards": [\n            {\n                "id": receipt["shard_id"],\n                "verify_exit": {\n                    "path": str(evidence / "verify-exit.json"),\n                    "sha256": provider.sha256(evidence / "verify-exit.json"),\n                },\n            }\n        ],\n    }\n    provider.save(evidence / "collection.json", spec)\n    catalog = scorer["build_catalog"](spec, root)\n    catalog["catalog_sha256"] = provider.sha256(evidence / "collection.json")\n    provider.save(evidence / "catalog.json", catalog)\n    result = scorer["aggregate_results"](catalog, root)\n    result["catalog_sha256"] = provider.sha256(evidence / "catalog.json")\n    provider.save(evidence / "results.json", result)\n    summary = {\n        "schema": "hung-cloud-cpu-receiver-v1",\n        "scope": "development_only",\n        "passed": True,\n        "executed": False,\n        "shard_id": receipt["shard_id"],\n        "jobs": len(receipt["job_ids"]),\n        "counts": result["counts"],\n        "artifact_sha256": receipt["artifact_sha256"],\n        "artifact_bytes": receipt["artifact_bytes"],\n        "original_receipt_sha256": provider.sha256(receipt_path),\n        "model_code_commit": expected_code,\n        "plan_sha256": provider.sha256(plan),\n        "partition_sha256": provider.sha256(partition),\n        "scorer_sha256": scorer_sha256,\n        "verification": terminal,\n        "elapsed_seconds": time.perf_counter() - started,\n        "results_sha256": provider.sha256(evidence / "results.json"),\n        "catalog_sha256": provider.sha256(evidence / "catalog.json"),\n        "peer_acceptance": "pending; technical restore/readback is not independent scientific review",\n    }\n    provider.save(evidence / "readback-summary.json", summary)\n    print(json.dumps(summary), flush=True)\n    return summary\n'
SCORER_SOURCE = '"""Read-only Development forecast scoring from SHA-pinned, verified artifacts.\n\nRecompute original-unit errors from stored issued forecasts and already arrived\nlabels. Never load Final inputs or choose configurations. All tables are partial\nuntil the separate coverage, fairness and independent-review gates pass.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport struct\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport torch\n\n\ndef file_sha(path: Path) -> str:\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef verified_file(pin: dict, root: Path) -> Path:\n    path = Path(pin["path"]).resolve()\n    if not path.is_relative_to(root.resolve()) or file_sha(path) != pin["sha256"]:\n        raise ValueError("Artifact outside project or SHA mismatch")\n    return path\n\n\ndef pin(path: Path, root: Path) -> dict:\n    path = path.resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("Artifact outside project")\n    return {"path": str(path), "sha256": file_sha(path)}\n\n\ndef method_family(job: dict) -> str:\n    suffix = "_" + job["cell"]\n    if not job["id"].endswith(suffix) or len(job["id"]) <= len(suffix):\n        raise ValueError("Job family/cell naming differs")\n    return job["id"][: -len(suffix)]\n\n\ndef build_catalog(spec: dict, root: Path) -> dict:\n    """Bind complete dispatcher receipts and VERIFIED_REUSE rows to read-only metrics."""\n    plan = json.loads(verified_file(spec["plan"], root).read_bytes())\n    partition = json.loads(verified_file(spec["partition"], root).read_bytes())\n    if plan["scope"] != "development_only" or partition["plan_sha256"] != spec["plan"]["sha256"]:\n        raise ValueError("Plan/partition binding differs")\n    cells = {c["id"]: c for c in plan["cells"]}\n    jobs = {j["id"]: j for j in plan["jobs"]}\n    configs = {c["id"]: c for c in plan["configs"]}\n    output = Path(spec.get("readback_output_root", plan["output_root"])).resolve()\n    rows = []\n\n    def entry(job_id, method, cell, manifest_pin, checkpoint_pin, source, target, proofs, status):\n        return {\n            "job_id": job_id,\n            "method": method,\n            "budget": cell["budget"],\n            "seed": cell["seed"],\n            "entity_id": cell["entity_id"],\n            "input_manifest_sha256": cell["input_manifest_sha256"],\n            "source_sha256": source["sha256"] if source else None,\n            "target_sha256": target["sha256"] if target else None,\n            "manifest": manifest_pin,\n            "checkpoint": checkpoint_pin,\n            "proofs": proofs,\n            "status": status,\n        }\n\n    if spec.get("reuse_ledger"):\n        ledger = json.loads(verified_file(spec["reuse_ledger"], root).read_bytes())\n        for row in ledger["jobs"]:\n            if row["status"] != "VERIFIED_REUSE":\n                continue\n            if not row["job_id"].startswith("main_"):\n                raise ValueError("Unsupported reuse mapping")\n            cell = cells[row["job_id"][len("main_") :]]\n            manifest_path = verified_file(row["run_manifest"], root)\n            manifest = json.loads(manifest_path.read_bytes())\n            checkpoint = {\n                "path": str(manifest_path.with_name("runner_checkpoint.pt")),\n                "sha256": manifest["final_runner_checkpoint_sha256"],\n            }\n            proofs = [spec["reuse_ledger"], spec["plan"], row["source"], row["target"]]\n            journal = manifest["event_journal"]\n            proofs.append(\n                {"path": str(manifest_path.with_name(journal["file"])), "sha256": journal["sha256"]}\n            )\n            rows.append(\n                entry(\n                    row["job_id"],\n                    "main",\n                    cell,\n                    row["run_manifest"],\n                    checkpoint,\n                    row["source"],\n                    row["target"],\n                    proofs,\n                    "VERIFIED_REUSE",\n                )\n            )\n    for completed in spec["completed_shards"]:\n        exit_record = json.loads(verified_file(completed["verify_exit"], root).read_bytes())\n        if exit_record["exit_code"] != 0:\n            raise ValueError("Shard verifier did not succeed")\n        shard = next(s for s in partition["shards"] if s["id"] == completed["id"])\n        for job_id in shard["job_ids"]:\n            job, cell = jobs[job_id], cells[jobs[job_id]["cell"]]\n            receipt_path = output / "receipts" / job_id / "job.json"\n            receipt = json.loads(receipt_path.read_bytes())\n            if (receipt["job_id"], receipt["plan_sha256"], receipt["shards_sha256"]) != (\n                job_id,\n                spec["plan"]["sha256"],\n                spec["partition"]["sha256"],\n            ):\n                raise ValueError("Job receipt identity differs")\n            config_pin = {k: configs[job["config"]][k] for k in ("path", "sha256")}\n            config = json.loads(verified_file(config_pin, root).read_bytes())\n            final_stage = f"replay-{config[\'replay\'][\'max_origins\']}"\n            proofs = [\n                spec["plan"],\n                spec["partition"],\n                completed["verify_exit"],\n                pin(receipt_path, root),\n                config_pin,\n            ]\n            for stage, sha in receipt["stages"].items():\n                stage_pin = {"path": str(receipt_path.with_name(stage + ".json")), "sha256": sha}\n                stage_record = json.loads(verified_file(stage_pin, root).read_bytes())\n                binding = stage_record["binding"]\n                if (\n                    binding["job"],\n                    binding["plan_sha256"],\n                    binding["shards_sha256"],\n                    binding["config_sha256"],\n                    binding["input_manifest_sha256"],\n                ) != (\n                    job,\n                    spec["plan"]["sha256"],\n                    spec["partition"]["sha256"],\n                    config_pin["sha256"],\n                    cell["input_manifest_sha256"],\n                ):\n                    raise ValueError("Stage provenance differs")\n                proofs.append(stage_pin)\n                for name, record in stage_record["files"].items():\n                    artifact = {\n                        "path": str(output / "jobs" / job_id / stage / name),\n                        "sha256": record["sha256"],\n                    }\n                    verified_file(artifact, root)\n                    proofs.append(artifact)\n            if final_stage not in receipt["stages"]:\n                raise ValueError("Final replay stage incomplete")\n            manifest_path = output / "jobs" / job_id / final_stage / "run_manifest.json"\n            manifest = json.loads(manifest_path.read_bytes())\n            checkpoint_name = (\n                "runner.pt"\n                if manifest["schema"] == "hung-single-baseline-replay-v1"\n                else "runner_checkpoint.pt"\n            )\n            # The final checkpoint hash is already bound by the stage inventory.\n            checkpoint = {\n                "path": str(manifest_path.with_name(checkpoint_name)),\n                "sha256": manifest["final_runner_checkpoint_sha256"],\n            }\n            for role in ("source", "target"):\n                if job[role]:\n                    proofs.append(job[role])\n            rows.append(\n                entry(\n                    job_id,\n                    method_family(job),\n                    cell,\n                    pin(manifest_path, root),\n                    checkpoint,\n                    job["source"],\n                    job["target"],\n                    proofs,\n                    "DISPATCH_VERIFIED",\n                )\n            )\n    return {\n        "scope": "development_only",\n        "schema": "hung-results-catalog-v1",\n        "spec": spec,\n        "runs": rows,\n    }\n\n\ndef metrics(days: list[dict]) -> dict | None:\n    if not days:\n        return None\n    hours = 24 * len(days)\n    return {\n        "hours": hours,\n        "mae_kwh": sum(d["absolute"] for d in days) / hours,\n        "rmse_kwh": math.sqrt(sum(d["squared"] for d in days) / hours),\n    }\n\n\ndef score_saved(state: dict, manifest: dict) -> dict:\n    """Score live predictions only; do not use candidate or audit performance."""\n    if state["schema"] not in {"hung-sprint2-runner-v1", "hung-single-baseline-runner-v1"}:\n        raise ValueError("Unsupported checkpoint schema")\n    if manifest.get("status") != "completed":\n        raise ValueError("Run not terminal")\n    scale = state["energy_scale"]\n    if not math.isfinite(scale) or scale <= 0:\n        raise ValueError("Invalid original-unit scale")\n    days = {}\n    issued_ids = set()\n    for issued in state["issued"]:\n        day_id = issued["day_id"]\n        if day_id in issued_ids:\n            raise ValueError("Duplicate issued day")\n        issued_ids.add(day_id)\n        labels = issued.get("y_scaled")\n        if labels is None:\n            if issued.get("normalized_mae") is not None:\n                raise ValueError("Missing labels carry a score")\n            continue\n        if issued.get("normalized_mae") is None:\n            raise ValueError("Arrived labels were not scored")\n        if state["schema"] == "hung-single-baseline-runner-v1":\n            prediction = issued["prediction"]\n        else:\n            levels = state["controller"]["levels"]\n            prediction = issued["forecasts"][levels.index(issued["active_level"])]\n        prediction, labels = prediction.reshape(-1).double(), labels.reshape(-1).double()\n        if prediction.shape != (24,) or labels.shape != (24,):\n            raise ValueError("Incomplete 24-hour horizon")\n        if not bool(torch.isfinite(prediction).all() and torch.isfinite(labels).all()):\n            raise ValueError("Nonfinite forecast/label")\n        if bool((prediction < 0).any() or (labels < 0).any()):\n            raise ValueError("Negative forecast/label")\n        error = (prediction - labels) * scale\n        days[day_id] = {\n            "absolute": float(error.abs().sum()),\n            "squared": float(error.square().sum()),\n            "label_sha256": hashlib.sha256(\n                struct.pack("<24d", *(labels * scale).tolist())\n            ).hexdigest(),\n        }\n    coverage = manifest["coverage"]\n    if len(issued_ids) != coverage["created"] or len(days) != coverage["scored"]:\n        raise ValueError("Checkpoint coverage differs")\n    calculated = metrics(list(days.values()))\n    declared = manifest["metrics_kwh"]\n    if isinstance(declared, dict) and "live" in declared:\n        declared = declared["live"]\n    if (calculated is None) != (declared is None):\n        raise ValueError("Declared metric availability differs")\n    if calculated is not None:\n        for key, value in calculated.items():\n            if not math.isclose(value, declared[key], rel_tol=1e-9, abs_tol=1e-9):\n                raise ValueError(f"Recomputed {key} differs")\n    return {"coverage": coverage, "metrics": calculated, "days": days}\n\n\ndef verify_artifacts(catalog: dict, root: Path) -> list[dict]:\n    if catalog.get("scope") != "development_only":\n        raise ValueError("Development-only catalog required")\n    records, identities = [], set()\n    for entry in catalog["runs"]:\n        if entry["status"] not in {"VERIFIED_REUSE", "DISPATCH_VERIFIED"}:\n            raise ValueError("Unmatched or unverified run cannot enter metric comparison")\n        identity = (entry["method"], entry["budget"], entry["seed"], entry["entity_id"])\n        if identity in identities:\n            raise ValueError("Duplicate logical run")\n        identities.add(identity)\n        for proof in entry["proofs"]:\n            verified_file(proof, root)\n        manifest = json.loads(verified_file(entry["manifest"], root).read_bytes())\n        checkpoint = verified_file(entry["checkpoint"], root)\n        if manifest["final_runner_checkpoint_sha256"] != entry["checkpoint"]["sha256"]:\n            raise ValueError("Manifest/checkpoint binding differs")\n        binding = manifest["input_binding"]\n        if binding["input_manifest_sha256"] != entry["input_manifest_sha256"]:\n            raise ValueError("Input binding differs")\n        if binding["entity_id"] != entry["entity_id"] or binding["history_days"] != entry["budget"]:\n            raise ValueError("Station/budget binding differs")\n        seed = manifest["config"].get("seed")\n        baseline_receipt_seed = (\n            seed is None\n            and entry["status"] == "DISPATCH_VERIFIED"\n            and entry["method"] in {"seasonal_naive", "frozen_source", "target_only"}\n        )\n        if (seed != entry["seed"] and not baseline_receipt_seed) or not entry["proofs"]:\n            raise ValueError("Seed or verification proof missing")\n        for role in ("source", "target"):\n            expected = entry.get(role + "_sha256")\n            if expected is not None and manifest.get(role + "_checkpoint_sha256") != expected:\n                raise ValueError("Source/Target binding differs")\n        state = torch.load(checkpoint, map_location="cpu", weights_only=True)\n        if state["binding"] != binding:\n            raise ValueError("Checkpoint input binding differs")\n        scored = score_saved(state, manifest)\n        records.append(\n            {\n                **entry,\n                **scored,\n                "code_commit": manifest["code"]["commit"],\n                "device": manifest.get("device"),\n                "final_stage_duration_seconds": manifest.get("duration_seconds"),\n                "cumulative_runtime_costs": manifest.get("runtime_costs", manifest.get("cost")),\n                "cumulative_proxy_costs": manifest.get("proxy_costs"),\n            }\n        )\n    return records\n\n\ndef paired_results(records: list[dict]) -> tuple[list[dict], list[dict]]:\n    cells = defaultdict(dict)\n    for record in records:\n        cells[(record["budget"], record["seed"], record["entity_id"])][record["method"]] = record\n    pairs, blocked = [], []\n    for (budget, seed, station), methods in sorted(cells.items()):\n        for reference in ("main", "target_only"):\n            if reference not in methods:\n                continue\n            baseline = methods[reference]\n            for method, candidate in sorted(methods.items()):\n                if method == reference:\n                    continue\n                reason = None\n                if candidate["input_manifest_sha256"] != baseline["input_manifest_sha256"]:\n                    reason = "input_mismatch"\n                for role in ("source_sha256", "target_sha256"):\n                    left, right = candidate.get(role), baseline.get(role)\n                    if left is not None and right is not None and left != right:\n                        reason = "initialization_mismatch"\n                common = sorted(candidate["days"].keys() & baseline["days"].keys())\n                if any(\n                    candidate["days"][d]["label_sha256"] != baseline["days"][d]["label_sha256"]\n                    for d in common\n                ):\n                    reason = "label_mismatch"\n                if not common:\n                    reason = reason or "no_paired_scored_days"\n                identity = {\n                    "budget": budget,\n                    "seed": seed,\n                    "entity_id": station,\n                    "method": method,\n                    "reference": reference,\n                }\n                if reason:\n                    blocked.append({**identity, "reason": reason})\n                    continue\n                left = metrics([candidate["days"][d] for d in common])\n                right = metrics([baseline["days"][d] for d in common])\n                pairs.append(\n                    {\n                        **identity,\n                        "paired_days": len(common),\n                        "method_metrics": left,\n                        "reference_metrics": right,\n                        "uses_source": candidate.get("source_sha256") is not None,\n                        "paired_day_ids_sha256": hashlib.sha256(\n                            json.dumps(common).encode()\n                        ).hexdigest(),\n                    }\n                )\n    return pairs, blocked\n\n\ndef station_first(pairs: list[dict]) -> list[dict]:\n    stations = defaultdict(list)\n    for pair in pairs:\n        key = (pair["budget"], pair["method"], pair["reference"], pair["entity_id"])\n        stations[key].append(pair)\n    groups = defaultdict(list)\n    for (budget, method, reference, station), rows in sorted(stations.items()):\n        means = {\n            side: {\n                metric: sum(r[side][metric] for r in rows) / len(rows)\n                for metric in ("mae_kwh", "rmse_kwh")\n            }\n            for side in ("method_metrics", "reference_metrics")\n        }\n        groups[(budget, method, reference)].append(\n            {\n                "entity_id": station,\n                "seeds": len(rows),\n                "uses_source": all(r["uses_source"] for r in rows),\n                **means,\n            }\n        )\n    result = []\n    for (budget, method, reference), rows in sorted(groups.items()):\n        averages = {\n            side: {\n                metric: sum(r[side][metric] for r in rows) / len(rows)\n                for metric in ("mae_kwh", "rmse_kwh")\n            }\n            for side in ("method_metrics", "reference_metrics")\n        }\n        gain = averages["reference_metrics"]["mae_kwh"] - averages["method_metrics"]["mae_kwh"]\n        denominator = averages["reference_metrics"]["mae_kwh"]\n        worse = sum(\n            r["method_metrics"]["mae_kwh"] > r["reference_metrics"]["mae_kwh"] for r in rows\n        ) / len(rows)\n        is_transfer = reference == "target_only" and all(r["uses_source"] for r in rows)\n        result.append(\n            {\n                "budget": budget,\n                "method": method,\n                "reference": reference,\n                "stations": len(rows),\n                "minimum_seeds_per_station": min(r["seeds"] for r in rows),\n                **averages,\n                "absolute_mae_gain_kwh": gain,\n                "relative_mae_gain": gain / denominator if denominator > 0 else None,\n                "worse_than_reference_station_fraction": worse,\n                "negative_transfer_station_fraction": worse if is_transfer else None,\n                "transfer_gain": gain / denominator if is_transfer and denominator > 0 else None,\n            }\n        )\n    return result\n\n\ndef aggregate_results(catalog: dict, root: Path) -> dict:\n    records = verify_artifacts(catalog, root)\n    pairs, blocked = paired_results(records)\n    return {\n        "schema": "hung-development-partial-results-v2",\n        "scope": "development_only",\n        "acceptance": "Provisional; full coverage and independent review required",\n        "runs": [{k: v for k, v in r.items() if k != "days"} for r in records],\n        "counts": {\n            "runs": len(records),\n            "scorable_runs": sum(bool(r["days"]) for r in records),\n            "zero_scored_runs": sum(not r["days"] for r in records),\n        },\n        "paired_cells": pairs,\n        "blocked_pairs": blocked,\n        "station_first": station_first(pairs),\n        "aggregation": "Complete hourly errors within paired station/seed; mean seeds within station, then equal station mean within each budget. Budgets remain separate. Transfer gain and negative transfer apply only to Source-using methods versus Target-only. Other comparisons report generic MAE differences/worse-than-reference fractions. Zero reference MAE gives undefined relative gain; absolute differences remain available.",\n    }\n\n\ndef main() -> None:\n    from experiments.hung_sprint2.dispatch_state import atomic_create\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--catalog", type=Path, required=True)\n    parser.add_argument("--catalog-sha256", required=True)\n    parser.add_argument("--project-root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument(\n        "--build-catalog", action="store_true", help="Input is a pinned collection specification"\n    )\n    args = parser.parse_args()\n    path = verified_file(\n        {"path": str(args.catalog), "sha256": args.catalog_sha256}, args.project_root\n    )\n    value = json.loads(path.read_bytes())\n    result = (\n        build_catalog(value, args.project_root)\n        if args.build_catalog\n        else aggregate_results(value, args.project_root)\n    )\n    result["catalog_sha256"] = args.catalog_sha256\n    if not args.output.resolve().is_relative_to(args.project_root.resolve()):\n        raise ValueError("Output outside project")\n    atomic_create(args.output, result)\n    print(\n        json.dumps(\n            {\n                "output": str(args.output),\n                "sha256": file_sha(args.output),\n                **result.get("counts", {}),\n            }\n        )\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
RECEIVER_SHA = "5cfd98dd7c4a64f672a12fce2d5a65454f069d5a2c6a099d3a3d7c7d9fb48571"
SCORER_SHA = "68fccbed056734d9b4842f7bda088880d7812d12417d64878d1aa018fc010456"
assert hashlib.sha256(RECEIVER_SOURCE.encode()).hexdigest() == RECEIVER_SHA
assert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest() == SCORER_SHA
receiver = types.ModuleType("pinned_cloud_receiver")
exec(compile(RECEIVER_SOURCE, "pinned_cloud_receiver", "exec"), receiver.__dict__)
scorer = {"__name__": "pinned_saved_forecast_scorer"}
exec(compile(SCORER_SOURCE, "pinned_saved_forecast_scorer", "exec"), scorer)
summary = receiver.receive(
    provider,
    scorer,
    archive,
    archive.parent / "receipt.json",
    repo,
    expected_code=MODEL_CODE,
    evidence=base / "cpu-readback",
    scorer_sha256=SCORER_SHA,
)
assert summary["passed"] and summary["executed"] is False and summary["jobs"] == 20
summary.update(
    {
        "receiver_code_commit": "3d777e3a2d03f805378b738192f879993e9538ad",
        "receiver_sha256": RECEIVER_SHA,
        "transport_helper_sha256": HELPER_SHA,
        "derivation": derivation,
        "torch": torch.__version__,
        "device": "cpu",
        "whole_receiver_seconds": time.perf_counter() - started,
    }
)
provider.save(base / "receiver-proof.json", summary)
print(json.dumps({"receiver_proof": summary}), flush=True)